# ETL  ·  Bronze → Stage  (`stage.defunciones`)

**Proyecto:** Plataforma Analítica de Mortalidad — PNUD / MSPAS
**Capa:** Bronze (crudo Delta) → **Stage** (conformado: limpio, tipado, estandarizado)
**Entorno:** Databricks · Serverless (read-only sobre Bronze, sin credenciales S3)

> Las reglas de limpieza de este notebook **NO** son genéricas: derivan de los hallazgos
> del EDA (`reporte_bronze_a_stage_v2_fase2.md`). Cada bloque cita la regla que aplica
> (`R-VALID-1`, `R-CARD-1`, `R-COMP-1`, `C1`–`C5`).

**Correcciones frente al código de ejemplo del enunciado:**
1. `norm_float` **antes** de unir (artefacto `"1.0"` del legacy — H2/C3).
2. `edad_anios` calculada con `Perdif` (días/meses/años/ignorado — H4/C2), no `Edadif` crudo.
3. `Sexo` dominio = {1,2}, **sin** el 9 (R-VALID-3).
4. **No** se nulifica `Caudef` por longitud: es 100% CIE-10 válido (H6).
5. Centinelas "Ignorado" (`9`/`999`) → `NULL` por columna según diccionario (R-COMP-1).

In [0]:
spark.sql("USE CATALOG workspace")

In [0]:
from datetime import datetime
_inicio = datetime.now()
_notebook_actual = "etl_bronze_to_stage"

In [0]:
# ── 0. SETUP ─────────────────────────────────────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("CREATE SCHEMA IF NOT EXISTS stage")

BRONZE = "bronze"
INE    = ["xlsx_ine", "sav_ine_legacy"]
META   = ["_anio", "_archivo_origen", "_fuente"]   # columnas de linaje, no se tocan

print("Schemas / tablas Bronze disponibles:")
for t in INE:
    print(f"  {BRONZE}.{t}: {spark.table(f'{BRONZE}.{t}').count():,} filas")

## 1. Normalización del artefacto float  ·  `R-VALID-1` / `C3`

El legacy 2015-2017 (`.sav`) guardó los códigos como float convertido a texto:
`"1.0"`, `"101.0"`. Si se une sin corregir, **el mismo municipio se parte en dos grupos**
(`"0101"` vs `"101.0"`) y `Sexo` casteado da valores erróneos.

`norm_float` quita el sufijo `.0` y manda `nan`/`none`/`""` → `NULL`.
Se ejecuta **antes** de unir y antes de cualquier agrupación (prerrequisito del k-anonimato).

In [0]:
# ── 1. NORMALIZACIÓN FLOAT (legacy) — R-VALID-1 / C3 ─────────────────────────
def norm_float(df):
    """Quita artefacto '.0' y normaliza centinelas textuales de vacío a NULL.
       No toca columnas de linaje (_anio, _archivo_origen, _fuente)."""
    for c in df.columns:
        if c.startswith("_"):
            continue
        col = F.col(f"`{c}`")
        df = df.withColumn(
            c,
            F.when(F.lower(F.trim(col)).isin("nan", "none", ""), None)
             .otherwise(F.regexp_replace(col, r"\.0$", ""))
        )
    return df

# xlsx (2018-2024) NO tiene Areag -> se agrega como NULL para unificar esquema (H1/D1)
xls = norm_float(spark.table(f"{BRONZE}.xlsx_ine")) \
        .withColumn("Areag", F.lit(None).cast("string"))
leg = norm_float(spark.table(f"{BRONZE}.sav_ine_legacy"))

# Unir por nombre de columna en orden idéntico (H1)
cols = xls.columns
df_unificado = xls.select(*cols).unionByName(leg.select(*cols))

print(f"Unificado: {df_unificado.count():,} filas  (esperado ~919,231)")

## 2. Conformación: tipado, geografía, fecha, edad, centinelas

- **`R-CARD-1`** — `lpad(municipio,4,'0')`: recupera ceros iniciales perdidos por el float.
- **`R-CONS-1/2`** — `fecha_ocurrencia` desde día/mes/año de **ocurrencia** (no registro).
- **`C2` (edad_anios)** — la corrección más importante: la unidad de `Edadif` depende de `Perdif`
  (1=días, 2=meses, 3=años, 9=ignorado→`Edadif=999`). Los ~46 mil infantes (días/meses)
  son **<1 año**, no "1-4 años".
- **`R-COMP-1`** — centinela "Ignorado" → `NULL` **por columna** según el diccionario:
  `9` en categóricas (`Puedif`, `Escodif`, `Asist`, `Ocur`, `Cerdef`, `Ecidif`);
  `999` en `Edadif`. **No** se nulifica `9` en `Mes` (=Septiembre) ni en `Día`.
- **`R-VALID-3`** — `Sexo` ∈ {1,2}; cualquier otro → `NULL`.
- **`periodo`** — marca PRE/POST-COVID (≤2019 vs ≥2020).

In [0]:
# ── 2. CONFORMACIÓN ──────────────────────────────────────────────────────────
sentinela_9 = ["Puedif", "Escodif", "Asist", "Ocur", "Cerdef", "Ecidif"]  # R-COMP-1

df_stage = (df_unificado
    # --- tipado base ---
    .withColumn("anio",  F.col("Añoocu").cast("int"))
    .withColumn("mes",   F.col("Mesocu").cast("int"))
    .withColumn("dia",   F.col("Diaocu").cast("int"))
    .withColumn("depto", F.col("Depocu").cast("int"))

    # --- geografía: lpad a 4 dígitos (R-CARD-1) ---
    .withColumn("muni_ocu", F.lpad(F.col("Mupocu"), 4, "0"))
    .withColumn("muni_reg", F.lpad(F.col("Mupreg"), 4, "0"))

    # --- Sexo dominio {1,2}, resto NULL (R-VALID-3) ---
    .withColumn("sexo",
        F.when(F.col("Sexo").isin("1", "2"), F.col("Sexo").cast("int"))
         .otherwise(None))

    # --- Causa: CIE-10 trim+upper, SIN filtrar por longitud (H6) ---
    .withColumn("caudef", F.trim(F.upper(F.col("Caudef"))))

    # --- fecha de ocurrencia validada (R-CONS-1) ---
    .withColumn("fecha_ocurrencia",
        F.expr("try_to_date(concat(lpad(`Diaocu`,2,'0'),'-',"
               "lpad(`Mesocu`,2,'0'),'-',`Añoocu`),'dd-MM-yyyy')"))

    # --- EDAD EN AÑOS usando Perdif (C2 — corrección clave) ---
    .withColumn("edad_anios",
        F.when(F.col("Perdif") == "9", None)                          # ignorado (Edadif=999)
         .when(F.col("Perdif") == "3", F.col("Edadif").cast("int"))   # 3 = años
         .when(F.col("Perdif").isin("1", "2"), F.lit(0))              # 1=días / 2=meses -> <1 año
         .otherwise(None))

    # --- centinela 'Ignorado' (9) -> NULL en categóricas (R-COMP-1) ---
    .withColumn("Puedif",  F.when(F.col("Puedif")  == "9", None).otherwise(F.col("Puedif")))
    .withColumn("Escodif", F.when(F.col("Escodif") == "9", None).otherwise(F.col("Escodif")))
    .withColumn("Asist",   F.when(F.col("Asist")   == "9", None).otherwise(F.col("Asist")))
    .withColumn("Ocur",    F.when(F.col("Ocur")    == "9", None).otherwise(F.col("Ocur")))
    .withColumn("Cerdef",  F.when(F.col("Cerdef")  == "9", None).otherwise(F.col("Cerdef")))
    .withColumn("Ecidif",  F.when(F.col("Ecidif")  == "9", None).otherwise(F.col("Ecidif")))

    # --- partición temporal pre/post COVID ---
    .withColumn("periodo",
        F.when(F.col("anio") <= 2019, F.lit("PRE_COVID"))
         .otherwise(F.lit("POST_COVID")))
)

# --- unicidad sin ID: no dropear, solo marcar (R-UNIQ-1) ---
# Se calcula DESPUES de cerrar la cadena anterior, sobre las columnas ya construidas.
cols_negocio_dedup = [c for c in df_stage.columns if not c.startswith("_")]
w_dedup = Window.partitionBy(*cols_negocio_dedup)
df_stage = df_stage.withColumn("dup_exacto", F.count("*").over(w_dedup) > 1)

## 3. Validez de dominios (filtros de rango)

Filtros **conservadores**, alineados con el EDA (Validez §3.3):
- `anio` ∈ [2015, 2024]  (rango del proyecto).
- `depto` ∈ [1, 22]  (22 departamentos de Guatemala).

No se filtra por `Sexo` ni por longitud de `Caudef` (ya tratados arriba).

In [0]:
# ── 3. FILTROS DE VALIDEZ ────────────────────────────────────────────────────
df_stage = (df_stage
    .filter(F.col("anio").between(2015, 2024))
    .filter(F.col("depto").between(1, 22))
)

print(f"Tras filtros de validez: {df_stage.count():,} filas")

## 4. Selección final y escritura Delta

Stage conserva **códigos crudos limpios** (`D2`): el etiquetado y las dimensiones
se construyen en el DW/Gold, no aquí. Se mantienen columnas de linaje.

Partición por `anio` (la columna temporal de ocurrencia, R-CONS-2).

In [0]:
# ── 4. SELECCIÓN FINAL Y ESCRITURA ───────────────────────────────────────────
df_final = df_stage.select(
    # temporales
    "anio", "mes", "dia", "fecha_ocurrencia", "periodo",
    # geografía (ocurrencia + registro)
    "depto", "muni_ocu", "muni_reg",
    # causa
    "caudef",
    # demografía
    "sexo", "edad_anios",
    F.col("Perdif").alias("perdif"),      # unidad de edad (NO etnia) — C1
    F.col("Puedif").alias("pueblo"),      # etnia real (9->NULL ya aplicado) — C1/H7
    F.col("Escodif").alias("escolaridad"),
    F.col("Ecidif").alias("estado_civil"),
    # lugar / certificación
    F.col("Ocur").alias("sitio_ocurrencia"),
    F.col("Asist").alias("asistencia"),
    F.col("Cerdef").alias("certificador"),
    F.col("Areag").alias("area"),         # urbano/rural — solo 2015-2017 (C4)
    # control
    "dup_exacto",
    # linaje (R-CONS-3)
    F.col("_anio").alias("lin_anio_archivo"),
    F.col("_archivo_origen").alias("lin_archivo"),
    F.col("_fuente").alias("lin_fuente"),
)

(df_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("anio")
    .saveAsTable("stage.defunciones"))

print(f"stage.defunciones: {spark.table('stage.defunciones').count():,} filas")

## 5. Checklist de aceptación (*Definition of Done*)

Valida automáticamente los puntos críticos del §9.6 del reporte.

In [0]:
# ── 5. CHECKLIST DE ACEPTACIÓN ───────────────────────────────────────────────
s = spark.table("stage.defunciones")
tot = s.count()

print("="*60)
print("CHECKLIST stage.defunciones")
print("="*60)

# 1. volumen
print(f"[1] filas totales: {tot:,}  (esperado ~919,231)")

# 2. sin sufijo .0 en columnas clave  +  Sexo en {1,2,NULL}
sexo_dom = [r["sexo"] for r in s.select("sexo").distinct().collect()]
print(f"[2] dominio sexo: {sorted([x for x in sexo_dom if x is not None])}  (esperado [1, 2])")

# 3. municipio longitud 4 en no-nulos
bad_muni = s.filter(F.col("muni_ocu").isNotNull() & (F.length("muni_ocu") != 4)).count()
print(f"[3] muni_ocu con longitud != 4: {bad_muni}  (esperado 0)")

# 4. edad_anios NULL exactamente donde perdif=9
n_p9   = s.filter(F.col("perdif") == "9").count()
n_edadnull_p9 = s.filter((F.col("perdif") == "9") & F.col("edad_anios").isNull()).count()
print(f"[4] perdif=9: {n_p9:,} | de ellos edad_anios NULL: {n_edadnull_p9:,}  (deben coincidir)")

# 5. caudef no vacío
n_caudef = s.filter(F.col("caudef").isNotNull() & (F.length("caudef") > 0)).count()
print(f"[5] caudef no nulo: {n_caudef:,}  ({100*n_caudef/tot:.1f}% — esperado ~100%)")

# 6. periodo bien repartido
s.groupBy("periodo").count().orderBy("periodo").show()

print("Checklist finalizado.")

In [0]:
spark.sql("SELECT COUNT(*) AS filas FROM workspace.stage.defunciones").show()
spark.sql("DESCRIBE workspace.stage.defunciones").show(50)

In [0]:
_fin = datetime.now()
_filas = spark.table("stage.defunciones").count()

spark.sql(f"""
INSERT INTO dw.etl_control_log (notebook, fecha_inicio, fecha_fin, estado, filas_salida, es_idempotente, nota)
VALUES (
    '{_notebook_actual}',
    '{_inicio}',
    '{_fin}',
    'EXITOSO',
    {_filas},
    true,
    'Overwrite completo: dup_exacto recalculado con Window por fila (no hardcoded)'
)
""")
print(f"Log registrado: {_notebook_actual} | {_filas:,} filas | {_fin - _inicio}")